# Simulating P vs. G

:::{sidebar} Discovering Game Theory
This course uses an **online textbook**: [Game Theory, a Discovery Approach](https://nordstrommath.com/IntroGameTheory2e/intro_game_theory.html), by Jennifer Nordstrom.
:::

Following on from the previous section, we would like to be able to construct a way to simulate the **Police vs. Guerrillas** game. The functions below will create the correct game matrix and will calculate the winning percentages of the various strategies. 

Based upon the winning percentages, the simulation will choose from the strategies that have performed best in the past. Using an **exponential moving average (ema)**, the winning percentages will be updated after each round of simulation. The convergence to the correct strategy mixture for each player will be sped up as the **ema** will emphasize recent results over the older, less accurate results.

The simulation functions are described below.

In [4]:
## Do not change this cell, only execute it. 
## This cell initializes Python so that pandas, numpy and scipy packages are ready to use.

import random
from IPython.display import display, Markdown
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import scipy as sp
import math
import scipy.stats as stats

## Setting up the Game
When playing Police vs. Guerrillas, we would like functions that create the correct game matrix given any values of

$$2\leq p,q \leq 20$$ 

where, as discussed [above](Z8.ipynb), we have:

$$g < p < 2g $$

We will initialize the values we need:
- $p$ : number of police units
- $g$ : number of guerrilla units

The player dictionaries will be created based upon the specific values of $p$ and $g$.

We will create 2 functions to accomplish the task:

1. **create_players_dict()**
2. **game_matrix()**

### Dictionaries

The function below creates player dictionaries and keeps track of the following:
- Row-Column indexes in the results table,
- Strategy numbers (integers), and
- Strategy names.

In [5]:
def create_player_dict(name, anchor_val):
    strat = np.arange(math.ceil(anchor_val / 2), anchor_val + 1)
    index = np.arange(len(strat))
    strat_name = [f"{x} - {anchor_val - x}" for x in strat]
    return {
        'player': name,
        'index': index,
        'strat': strat,
        'strat_name': strat_name
    }

We will initialize the values for
- $p$
- $g$
- $\alpha$

Then we will create both player dictionaries.

In [6]:
p = 8
g = 7
# Creating the dictionaries
P = create_player_dict('P', p)
G = create_player_dict('G', g)
# Uncomment the lines below to see the structures we have created
# print(P)
# print(G)

### Game Matrix

Now that we have the player dictionaries as well as the values $p$ and $g$ initialized, the following function will create a game matrix with outcomes all set to $0.0$.

First, let's verify the values of $p$ and $g$ we are working with.

In [7]:
print(p)
print(g)

8
7


The following code creates the contours of our game matrix with interior values set equal to $0.0$.

In [8]:
def game_matrix(player1, player2):
   # Create an empty DataFrame with specified index and columns
   res = pd.DataFrame(0.0, index=player1['strat_name'], columns=player2['strat_name'])
   return res

In [9]:
results = game_matrix(P,G)
results

,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.0,0.0,0.0,0.0
5 - 3,0.0,0.0,0.0,0.0
6 - 2,0.0,0.0,0.0,0.0
7 - 1,0.0,0.0,0.0,0.0
8 - 0,0.0,0.0,0.0,0.0


We can access the $(i,j)$ entry in the matrix above using the pandas method 
<center><span style='color:blue'>.iloc[i,j]</span></center></br>

We can also use <center><span style='color:blue'>.loc[P['strat_name'][i],G['strat_name'][j]]</span></center></br>

if we know the column strategy or row strategy indentifier.

### Init() Function

The interior entries of the **results** dataframe will be winning percentages of the row and column strategies when played against one another.

To initialize these values, we begin by playing each column strategy against each row strategy. We plan for 40 simulated battles with $10$ **ghost results**.  With $5$ ghost wins and $5$ ghost losses, the win percentages will skew toward the center (e.g. 0.5). The **ghost results** approach is typical when we wish to avoid seeding the simulation with extreme values, in this case, zeroes and ones.

With $10$ ghost results included, the simulation results appear as though $50$ total results were simulated. The **init()** function code is given below.

In [10]:
def init(p_max, g_max, p_strat, g_strat):
   A_0 = p_strat ; B_0 = p_max - A_0
   A = np.random.choice([A_0, B_0])
   B = p_max - A ;
   res = ()
   for k in range(40):
      a_0 = g_strat ; b_0 = g_max - a_0
      a = np.random.choice([a_0, b_0])
      b = g_max - a
      if ( ( a > A ) or ( b > B ) ):
         v = 0
      else:
         v = 1
      res = np.append(res, v)
   temp = ( sum(res) + 5 ) / 50
   t = round(temp,3)
   ## uncomment line below to help with debugging the init() function.
   #   print(f"police play ({A},{B}) while guerrillas play ({a},{b})")
   return t   

The function above will be used in a nested for-loop to calculate the needed values and assign them to the correct entry in the **results** dataframe. The code below accomplishes the initialization of the **results** table created above.

In [104]:
p = 8
g = 7

for i in P['index']:
   ##print(f"Processing row {i}...") # Debugging line
   for j in G['index']:
      results.loc[P['strat_name'][i],G['strat_name'][j]] = init(p,g, int(P['strat_name'][i][0]),int(G['strat_name'][j][0]))

results

,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.90,0.10,0.10,0.10
5 - 3,0.46,0.54,0.10,0.10
6 - 2,0.10,0.44,0.52,0.10
7 - 1,0.10,0.10,0.50,0.54
8 - 0,0.10,0.10,0.10,0.46


In [105]:
def row_column():
  # Calculate Column Weights and Row Weights
  col_means = results.mean()
  col_probs = col_means / col_means.sum()
  row_means = results.mean(axis=1)
  row_probs = row_means / row_means.sum()
  # Choose a row and column based on average win percentage
  chosen_col = np.random.choice(results.columns, p=col_probs)
  chosen_row = np.random.choice(results.index, p=row_probs)
  return chosen_row , chosen_col

In [106]:
def ema(current_ema, new_result, alpha):
  new_ema = (new_result * alpha) + (current_ema * (1 - alpha))
  return new_ema

In [107]:
results

,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.90,0.10,0.10,0.10
5 - 3,0.46,0.54,0.10,0.10
6 - 2,0.10,0.44,0.52,0.10
7 - 1,0.10,0.10,0.50,0.54
8 - 0,0.10,0.10,0.10,0.46


In [108]:
# Verifying that p, g, and alpah are set correctly.
print(p)
print(g)
print(alpha)

8
7
0.05


The play function does not appear to be working correctly right now.

In [113]:
def play(p,g, alpha):
  r, c = row_column()
  p_0 = int(r[0])
  G = int(c[0])
  A = np.random.choice([p_0, p - p_0]) ; B = p - A ;
  F = g - G ;
  if (G > A) or (F > B):
    v = 0
  else:
    v = 1
  old = results.loc[r,c]
  new = ema(old, val, alpha) 
  results.loc[r][c] = new
  return r, c, new

In [114]:
row, column, winloss = play(8,7,0.05)
print(row)
print(column)
print(winloss)
results

5 - 3
7 - 0
0.045438205500000016


,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.282430,0.038742,0.025419,0.038742
5 - 3,0.198015,0.318865,0.012158,0.045438
6 - 2,0.033124,0.090592,0.163182,0.053144
7 - 1,0.016677,0.038742,0.193710,0.169458
8 - 0,0.065610,0.028243,0.065610,0.160392


In [115]:
reps = 25 ; alpha = 0.2
for k in range(reps):
  play(8,7,0.1)

reps = 50 ; alpha = 0.1
for k in range(reps):
  play(8,7,0.1)
    
reps = 125 ; alpha = 0.05
for k in range(reps):
  play(8,7,0.1)

results

,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.071790,0.009848,0.008863,0.016677
5 - 3,0.040770,0.111181,0.003815,0.007578
6 - 2,0.014259,0.035097,0.046087,0.031381
7 - 1,0.006461,0.012158,0.067543,0.053178
8 - 0,0.038742,0.010942,0.031381,0.076715
